In [1]:
import sys

PROJECT_ROOT = "/workspace/nyc-building-risk"
COMMON_PATH = f"{PROJECT_ROOT}/spark/common"

if COMMON_PATH not in sys.path:
    sys.path.insert(0, COMMON_PATH)

from spark_session import create_spark_session
from minio_config import minio_path

spark = create_spark_session(
    "NYC Building Risk - Silver PLUTO Test"
)

print("Spark version:", spark.version)
print("Spark UI:", spark.sparkContext.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/01 20:03:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/01 20:03:37 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/09/01 20:03:37 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Spark version: 3.4.0
Spark UI: http://ff7529cf1dd6:4042


In [2]:
pluto_file = minio_path(
    "bronze/pluto/"
    "snapshots/"
    "version=26v2/"
    "page_00001.json"
)

print("Reading:")
print(pluto_file)

Reading:
s3a://nyc-building-risk/bronze/pluto/snapshots/version=26v2/page_00001.json


In [3]:
pluto_df = (
    spark.read
    .option("multiline", "true")
    .json(pluto_file)
)

print("Rows:", pluto_df.count())

26/09/01 20:04:12 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Rows: 10000


In [4]:
pluto_df.printSchema()

root
 |-- address: string (nullable = true)
 |-- affresfar: string (nullable = true)
 |-- appbbl: string (nullable = true)
 |-- appdate: string (nullable = true)
 |-- areasource: string (nullable = true)
 |-- assessland: string (nullable = true)
 |-- assesstot: string (nullable = true)
 |-- bbl: string (nullable = true)
 |-- bct2020: string (nullable = true)
 |-- bctcb2020: string (nullable = true)
 |-- bldgarea: string (nullable = true)
 |-- bldgclass: string (nullable = true)
 |-- bldgdepth: string (nullable = true)
 |-- bldgfront: string (nullable = true)
 |-- block: string (nullable = true)
 |-- borocode: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- bsmtcode: string (nullable = true)
 |-- builtfar: string (nullable = true)
 |-- cb2010: string (nullable = true)
 |-- cd: string (nullable = true)
 |-- comarea: string (nullable = true)
 |-- commfar: string (nullable = true)
 |-- condono: string (nullable = true)
 |-- council: string (nullable = true)
 |-- ct2010

In [5]:
from pyspark.sql import functions as F


pluto_df.select(
    "bbl",
    "borocode",
    "borough",
    "block",
    "lot",
    "address",
    "landuse",
    "bldgclass",
    "yearbuilt",
    "unitsres",
    "unitstotal",
    "lotarea",
    "bldgarea",
    "latitude",
    "longitude",
    "version"
).show(
    20,
    truncate=False
)


print(
    "Rows:",
    pluto_df.count()
)


print(
    "Missing BBL:",
    pluto_df
    .filter(
        F.col("bbl").isNull()
        | (F.trim(F.col("bbl")) == "")
    )
    .count()
)


print(
    "BBL already 10 digits:",
    pluto_df
    .filter(
        F.trim(F.col("bbl")).rlike(
            "^[0-9]{10}$"
        )
    )
    .count()
)


print(
    "BBL with decimal suffix:",
    pluto_df
    .filter(
        F.trim(F.col("bbl")).rlike(
            "^[0-9]{10}\\.0+$"
        )
    )
    .count()
)


print(
    "Duplicate BBL:",
    pluto_df.count()
    - pluto_df.select("bbl").distinct().count()
)


print(
    "Missing coordinates:",
    pluto_df
    .filter(
        F.col("latitude").isNull()
        | F.col("longitude").isNull()
    )
    .count()
)


print(
    "Missing yearbuilt:",
    pluto_df
    .filter(
        F.col("yearbuilt").isNull()
        | (F.trim(F.col("yearbuilt")) == "")
    )
    .count()
)


print("Versions:")
pluto_df.select(
    "version"
).distinct().show(
    truncate=False
)

+-------------------+--------+-------+-----+----+--------------------+-------+---------+---------+--------+----------+-------+--------+----------+-----------+-------+
|bbl                |borocode|borough|block|lot |address             |landuse|bldgclass|yearbuilt|unitsres|unitstotal|lotarea|bldgarea|latitude  |longitude  |version|
+-------------------+--------+-------+-----+----+--------------------+-------+---------+---------+--------+----------+-------+--------+----------+-----------+-------+
|1000010010.00000000|1       |MN     |1    |10  |140 CARDER ROAD     |8      |Y4       |1900     |0       |1         |7577714|2532066 |40.6887632|-74.0187179|26v2   |
|1000010100.00000000|1       |MN     |1    |100 |CARDER ROAD         |8      |Y4       |0        |0       |1         |23121  |10000   |40.6927301|-74.0138617|26v2   |
|1000010101.00000000|1       |MN     |1    |101 |1 LIBERTY ISLAND    |8      |P7       |1900     |0       |0         |541886 |541886  |40.6899196|-74.0453371|26v2   

Rows: 10000


Missing BBL: 0
BBL already 10 digits: 0


BBL with decimal suffix: 10000


Duplicate BBL: 0


Missing coordinates: 79


Missing yearbuilt: 20
Versions:
+-------+
|version|
+-------+
|26v2   |
+-------+



In [6]:
spark = create_spark_session(
    "NYC Building Risk - PLUTO Silver Validation"
)

26/09/01 20:15:18 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [7]:
silver_pluto_path = minio_path(
    "silver/pluto/version=26v2"
)

pluto_silver_df = (
    spark.read
    .parquet(silver_pluto_path)
)

print(
    "PLUTO Silver rows:",
    pluto_silver_df.count()
)

PLUTO Silver rows: 858284


In [8]:
from pyspark.sql import functions as F

pluto_silver_df.select(
    "bbl",
    "borough",
    "address",
    "yearbuilt",
    "unitsres",
    "unitstotal",
    "latitude",
    "longitude",
    "snapshot_version"
).show(
    20,
    truncate=False
)

print(
    "Distinct BBL:",
    pluto_silver_df
    .select("bbl")
    .distinct()
    .count()
)

+----------+---------+--------------------+---------+--------+----------+----------+-----------+----------------+
|bbl       |borough  |address             |yearbuilt|unitsres|unitstotal|latitude  |longitude  |snapshot_version|
+----------+---------+--------------------+---------+--------+----------+----------+-----------+----------------+
|1000010010|MANHATTAN|140 CARDER ROAD     |1900     |0       |1         |40.6887632|-74.0187179|26v2            |
|1000010100|MANHATTAN|CARDER ROAD         |null     |0       |1         |40.6927301|-74.0138617|26v2            |
|1000010101|MANHATTAN|1 LIBERTY ISLAND    |1900     |0       |0         |40.6899196|-74.0453371|26v2            |
|1000010111|MANHATTAN|ANDES ROAD          |null     |0       |1         |40.6929217|-74.0176373|26v2            |
|1000010112|MANHATTAN|ANDES ROAD          |null     |0       |1         |40.6929271|-74.0183477|26v2            |
|1000010150|MANHATTAN|COMFORT ROAD        |null     |0       |1         |40.6876026|-74.

Distinct BBL: 858284
